In [2]:
import pandas as pd

In [23]:
from pathlib import Path
from zipfile import ZipFile
import csv
import io

data_dir = Path(r"C:\Users\user\Documents\LAB\kci\data")
output_path = Path(r"C:\Users\user\Documents\LAB\kci\따릉이데이터_2025.csv")

zip_files = sorted(data_dir.glob("tpss_bcycl_od_statnhm_2025??.zip"))

rename_map = {
    "전체_건수": "전체건수",
    "기준_시간대": "기준_시간",
    "시작_대여소": "시작_대여소명",
    "종료_대여소": "종료_대여소명",
}

expected_columns = None

with output_path.open(
    "w",
    encoding="utf-8-sig",
    newline="",
) as output_file:

    writer = csv.writer(output_file, lineterminator="\n")

    for zip_path in zip_files:
        print(f"{zip_path.name} 병합 시작")

        with ZipFile(zip_path) as zip_file:
            csv_files = sorted(
                name for name in zip_file.namelist()
                if name.lower().endswith(".csv")
            )

            for csv_name in csv_files:
                with zip_file.open(csv_name) as raw_file:
                    # 헤더를 이용해 파일별 인코딩 판별
                    header_bytes = raw_file.readline()

                    try:
                        header_text = header_bytes.decode("utf-8-sig")
                        encoding = "utf-8"
                    except UnicodeDecodeError:
                        header_text = header_bytes.decode("cp949")
                        encoding = "cp949"

                    original_columns = next(csv.reader([header_text]))

                    columns = [
                        rename_map.get(column, column)
                        for column in original_columns
                    ]

                    if expected_columns is None:
                        expected_columns = columns
                        writer.writerow(columns)
                    elif columns != expected_columns:
                        raise ValueError(
                            f"열 구성이 다릅니다: {csv_name}"
                        )

                    # 이미 헤더를 읽었으므로 나머지 데이터만 병합
                    with io.TextIOWrapper(
                        raw_file,
                        encoding=encoding,
                        newline="",
                    ) as input_file:

                        last_character = "\n"

                        while True:
                            text = input_file.read(1024 * 1024)

                            if not text:
                                break

                            output_file.write(text)
                            last_character = text[-1]

                        if last_character not in "\r\n":
                            output_file.write("\n")

        print(f"{zip_path.name} 병합 완료")

print(f"최종 파일: {output_path}")

tpss_bcycl_od_statnhm_202501.zip 병합 시작
tpss_bcycl_od_statnhm_202501.zip 병합 완료
tpss_bcycl_od_statnhm_202502.zip 병합 시작
tpss_bcycl_od_statnhm_202502.zip 병합 완료
tpss_bcycl_od_statnhm_202503.zip 병합 시작
tpss_bcycl_od_statnhm_202503.zip 병합 완료
tpss_bcycl_od_statnhm_202504.zip 병합 시작
tpss_bcycl_od_statnhm_202504.zip 병합 완료
tpss_bcycl_od_statnhm_202505.zip 병합 시작
tpss_bcycl_od_statnhm_202505.zip 병합 완료
tpss_bcycl_od_statnhm_202506.zip 병합 시작
tpss_bcycl_od_statnhm_202506.zip 병합 완료
tpss_bcycl_od_statnhm_202507.zip 병합 시작
tpss_bcycl_od_statnhm_202507.zip 병합 완료
tpss_bcycl_od_statnhm_202508.zip 병합 시작
tpss_bcycl_od_statnhm_202508.zip 병합 완료
tpss_bcycl_od_statnhm_202509.zip 병합 시작
tpss_bcycl_od_statnhm_202509.zip 병합 완료
tpss_bcycl_od_statnhm_202510.zip 병합 시작
tpss_bcycl_od_statnhm_202510.zip 병합 완료
tpss_bcycl_od_statnhm_202511.zip 병합 시작
tpss_bcycl_od_statnhm_202511.zip 병합 완료
tpss_bcycl_od_statnhm_202512.zip 병합 시작
tpss_bcycl_od_statnhm_202512.zip 병합 완료
최종 파일: C:\Users\user\Documents\LAB\kci\따릉이데이터_2025.csv


In [3]:
df = pd.read_csv(r'C:\Users\user\Documents\LAB\kci\따릉이데이터_2025.csv')

In [5]:
df.head()

,기준_날짜,집계_기준,기준_시간,시작_대여소_ID,시작_대여소명,종료_대여소_ID,종료_대여소명,전체건수,전체_이용_분,전체_이용_거리
0,20250109,출발시간,1950,ST-2056,발산1동_035_2,ST-1064,발산1동_056_1,1,4.0,680.0
1,20250109,출발시간,930,ST-1474,공릉1동_031_1,ST-3060,공릉2동_050_1,1,8.0,1240.0
2,20250109,출발시간,2050,ST-717,중앙동_028_1,ST-725,은천동_022_1,1,8.0,1105.0
3,20250109,출발시간,1615,ST-3017,합정동_025_1,ST-8,합정동_023_2,1,5.0,735.0
4,20250109,출발시간,2200,ST-158,이화동_009_1,ST-159,"종로5,6가동_002_1",1,7.0,652.0


In [17]:
df.shape

(68023993, 10)

In [7]:
df["기준_날짜"] = (
    pd.to_datetime(df["기준_날짜"].astype(str))
      .dt.normalize()
)

In [ ]:
# 날짜 형식 및 시간을 00:00:00으로 통일
dates = pd.to_datetime(df["기준_날짜"], errors="coerce").dt.normalize()

# 2025년 전체 날짜
all_dates_2025 = pd.date_range("2025-01-01", "2025-12-31", freq="D")

# df에 없는 날짜
missing_dates = all_dates_2025.difference(dates.dropna().unique())  

print("없는 날짜 수:", len(missing_dates))
display(pd.DataFrame({
    "없는_날짜": missing_dates,
    "날짜_문자열": missing_dates.strftime("%Y%m%d")
}))

없는 날짜 수: 10


,없는_날짜,날짜_문자열
0,2025-01-01,20250101
1,2025-01-02,20250102
2,2025-01-03,20250103
3,2025-01-04,20250104
4,2025-01-05,20250105
5,2025-01-06,20250106
6,2025-01-07,20250107
7,2025-01-08,20250108
8,2025-07-01,20250701
9,2025-07-02,20250702


In [14]:
pack = pd.read_csv(r'C:\Users\user\Documents\LAB\kci\data\서울시 공공자전거 따릉이 대여소 마스터 정보.csv' , encoding='cp949')

In [15]:
pack.head()

,대여소_ID,주소1,주소2,위도,경도
0,ST-999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.000000,0.000000
1,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.000000,0.000000
2,ST-997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,37.534390,126.869598
3,ST-996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,37.524334,126.850548
4,ST-995,서울특별시 양천구 중앙로 153 공중화장실,NaN,37.510597,126.857323


In [16]:
# 위도 또는 경도가 NaN이거나 0인 행
invalid_pack = pack[
    pack[["위도", "경도"]].isna().any(axis=1)
    | pack[["위도", "경도"]].eq(0).any(axis=1)
]

display(invalid_pack)
print("해당 행 수:", len(invalid_pack))

,대여소_ID,주소1,주소2,위도,경도
0,ST-999,서울특별시 양천구 목동서로 280,목동아파트 8단지 상가동,0.0,0.0
1,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.0,0.0
11,ST-989,서울특별시 마포구 월드컵로5길 11,합정동 주민센터,0.0,0.0
156,ST-858,서울특별시 강북구 덕릉로40길 48,번동 619-53,0.0,0.0
171,ST-844,서울특별시 성북구 인촌로 16,성북구 안암동2가 140-1,0.0,0.0
...,...,...,...,...,...
3353,ST-1060,서울특별시 강동구 둔촌동 산 125,일자산 체육관,0.0,0.0
3386,ST-1030,서울특별시 은평구 녹번동 141-92,유성빌딩 옆,0.0,0.0
3394,ST-1022,서울특별시 은평구 진관동 2-6,입곡삼거리,0.0,0.0
3409,ST-1009,서울특별시 양천구 신월동 263,신월 청소년 문화센터,0.0,0.0


해당 행 수: 77
